In [3]:

import numpy as np
from matplotlib import pyplot as plt
from utils import imgio as iio
import os
import shutil
import glob

import npy2bdv
import sys 
import re
import wx
import subprocess

import pathlib as pl


In [55]:
# Path to the folder containing the timelapse images
timelapse_path = r'g:\2PM\Amandine\250408_Doc1_11-01-54'
# Path to the folder containing the static images
static_path = r'g:\2PM\Amandine\250408_Doc1_11-01-55'


align_channel_suffix = '6RED'
# list files in static folder

static_files = glob.glob(static_path + f'/*{align_channel_suffix}*.tif')
static_files.sort()
print(static_files)

# list files in timelapse folder


timelapse_files = glob.glob(timelapse_path + f'/*{align_channel_suffix}*.tif')
timelapse_files.sort()
print(timelapse_files)


['g:\\2PM\\Amandine\\250408_Doc1_11-01-55\\11-01-55_Doc1_PMT - PMT [6RED] _C02_Time Time0000.ome.tif']
['g:\\2PM\\Amandine\\250408_Doc1_11-01-54\\11-01-54_Doc1_PMT - PMT [6RED] _C02_Time Time0000.ome.tif', 'g:\\2PM\\Amandine\\250408_Doc1_11-01-54\\11-01-54_Doc1_PMT - PMT [6RED] _C02_Time Time0001.ome.tif', 'g:\\2PM\\Amandine\\250408_Doc1_11-01-54\\11-01-54_Doc1_PMT - PMT [6RED] _C02_Time Time0002.ome.tif', 'g:\\2PM\\Amandine\\250408_Doc1_11-01-54\\11-01-54_Doc1_PMT - PMT [6RED] _C02_Time Time0003.ome.tif', 'g:\\2PM\\Amandine\\250408_Doc1_11-01-54\\11-01-54_Doc1_PMT - PMT [6RED] _C02_Time Time0004.ome.tif', 'g:\\2PM\\Amandine\\250408_Doc1_11-01-54\\11-01-54_Doc1_PMT - PMT [6RED] _C02_Time Time0005.ome.tif', 'g:\\2PM\\Amandine\\250408_Doc1_11-01-54\\11-01-54_Doc1_PMT - PMT [6RED] _C02_Time Time0006.ome.tif', 'g:\\2PM\\Amandine\\250408_Doc1_11-01-54\\11-01-54_Doc1_PMT - PMT [6RED] _C02_Time Time0007.ome.tif', 'g:\\2PM\\Amandine\\250408_Doc1_11-01-54\\11-01-54_Doc1_PMT - PMT [6RED] _C02_Ti

In [4]:
def make_bat(bat_dir, ds_name, idx, proc_bat_fn=None):
    bat_file = bat_dir / f'proc_{idx}.bat'
    
    t = '@echo off\n'
    
    t += f'set dsname={ds_name}\n'
    t += f'set workdir={bat_dir}\n'
    
    # make full path to scrip as parent folder of the bat_dir
    proc_bat_fn = proc_bat_fn or 'proc.bat'
    proc_path = bat_dir.parent / proc_bat_fn
    t += f'call {proc_path}\n'
    
    with open(bat_file, 'w') as f:
        f.write(t)
        
    return bat_file

In [5]:
def make_repos_bat(bat_dir, ds_name, repos_bat_fn=None):
    bat_file = bat_dir / f'repos.bat'
    
    t = '@echo off\n'
    
    t += f'set dsname={ds_name}\n'
    t += f'set workdir={bat_dir}\n'
    t += f'set posfile={bat_dir / "scanPos.txt"}\n'
    
    # make full path to scrip as parent folder of the bat_dir
    repos_bat_fn = repos_bat_fn or 'repos.bat'
    proc_path = bat_dir.parent / repos_bat_fn
    t += f'call {proc_path}\n'
    
    with open(bat_file, 'w') as f:
        f.write(t)
        
    return bat_file

# v0.1

In [6]:
# v0.1: red chennel only
# ToDo: Copy-paste and make v1 which loops all listed channels in all copy + use m-ch configs
assert len(static_files) == 1, 'There should be only one static image'

static_file = static_files[0]

# make tmp dir
tmp_path = pl.Path(timelapse_files[0]).parent.parent / 'tmp'
if tmp_path.exists():
    shutil.rmtree(tmp_path)
    
tmp_path.mkdir()

# make name templates:

static_file_frame = r'00-00-00_Doc1_PMT - PMT [6RED] _C02_Time Time0000.ome.tif'
timela_file_frame = r'00-00-00_Doc1_PMT - PMT [6RED] _C02_Time Time0001.ome.tif'

coords = ''
for idx, timelapse_file in enumerate(timelapse_files):
    # make dir for each timepoint
    
    ds_name = f'{idx:06d}_Doc1_00-00-00'
    timepoint_path = tmp_path / ds_name
    timepoint_path.mkdir()
    
    # copy static file
    shutil.copy(static_file, timepoint_path / static_file_frame)
    
    # copy timelapse file
    shutil.copy(timelapse_file, timepoint_path / timela_file_frame)
    
    # make bat file 
    bat_file = make_bat(tmp_path, ds_name, idx)
    
    
    # run bat file and wait for it to finish
    p = subprocess.Popen(str(bat_file), shell=True)
    p.wait()
    
    # read the file with the coordinates at the timepoint_path+'_A' / 'algn' / 'tile_000' / 'scanPos.txt'
    pos_file = tmp_path / f'{ds_name}_A' / 'algn' / 'tile_000' / 'scanPos.txt'
    with open(pos_file, 'rt') as f:
        lines = f.readlines()
    
    coords += lines[1]
    
    
# add the coordinates of the sttaic file (lines[0]) to the end of the coords
coords += lines[0]

# save the coords to a file in the tmp_path
coords_file = tmp_path / 'scanPos.txt'
with open(coords_file, 'wt') as f:
    f.write(coords)
    
    
# make joined dataset 000000_Doc1_00-00-01
ds_name_all = '000000_Doc1_00-00-01'
timela_file_frame_all = r'00-00-01_Doc1_PMT - PMT [6RED] _C02_Time Time%04d.ome.tif'



# go through all the timepoints and move timelapse files to the new folder
all_timepoint_path = tmp_path / ds_name_all
all_timepoint_path.mkdir()

for idx, timelapse_file in enumerate(timelapse_files):
    ds_name = f'{idx:06d}_Doc1_00-00-00'
    timepoint_path = tmp_path / ds_name
    
    shutil.move(timepoint_path / timela_file_frame, str(all_timepoint_path / timela_file_frame_all) % idx)

last_idx = len(timelapse_files)
    
# move static file to the new folder as well, the last one
shutil.move(timepoint_path / static_file_frame, str(all_timepoint_path / timela_file_frame_all) % (last_idx))

# create reposotitioning bat file
repos_bat = make_repos_bat(tmp_path, ds_name_all)

# run the repos.bat file
p = subprocess.Popen(str(repos_bat), shell=True)
p.wait()

# in the end one might want to move the sttatic timeframe to replicate the timelapse one at each timepoint as an addiitonal channel
out_path = all_timepoint_path.parent /(all_timepoint_path.stem+'_A')
all_tf = True

if all_tf:
    static_tf_last = out_path / (timela_file_frame_all % (last_idx))
    static_file_frame_all = timela_file_frame_all.replace(' _C0', ' _C1')
    for idx, timelapse_file in enumerate(timelapse_files):
        # copy static file (out_path / timela_file_frame_all) % (last_idx) to each timepoint
        shutil.copy(static_tf_last, out_path / (static_file_frame_all % idx))
        
    # remove the last file  -static_tf_last
    os.remove(static_tf_last)
        
# move the whole out_path folder to the final destination
timelapse_path_pl = pl.Path(timelapse_path)
final_path = timelapse_path_pl.parent / (timelapse_path_pl.stem + '_aligned')

if final_path.exists():
    print(f'{final_path} already exists, aborting'
          f'Please remove it first and rerun the script'
          f'You can find the aligned data in {out_path}')
else:
    shutil.move(out_path, final_path)
    print(f'Aligned data saved to {final_path}')
    
    #remove the tmp folder
    shutil.rmtree(tmp_path)

Aligned data saved to g:\2PM\Amandine\240827_Doc1_00-11-00_aligned


# v1

In [20]:
# v1 [WIP]: Blue, Green & Red channels, alignebt by red 
# ToDo: 
#    + Copy-paste and 
#    + make v1 which 
#    + loops all listed channels 
#    + in all copy 
#    + use m-ch configs
assert len(static_files) == 1, 'There should be only one static image'

static_file = static_files[0]

# make tmp dir
tmp_path = pl.Path(timelapse_files[0]).parent.parent / 'tmp'
if tmp_path.exists():
    shutil.rmtree(tmp_path)
    
tmp_path.mkdir()

channels_sfx = {
    0: '[2BLUE] _C00_Time', 
    1: '[5GREEN] _C01_Time', 
    2: '[6RED] _C02_Time',
    3: '[7FarRED] _C03_Time',
}
all_channels = list(channels_sfx.keys())
# make name templates:
# static_file_frame = r'00-00-00_Doc1_PMT - PMT [6RED] _C02_Time Time0000.ome.tif'
# timela_file_frame = r'00-00-00_Doc1_PMT - PMT [6RED] _C02_Time Time0001.ome.tif'

ref_ch = 2
tmpl_file_prfx = '00-00-00_Doc1_PMT - PMT '
static_file_frame_sfx = ' Time0000.ome.tif'
timela_file_frame_sfx = ' Time0001.ome.tif'


coords = ''
for idx, timelapse_file in enumerate(timelapse_files):
    # make dir for each timepoint
    
    ds_name = f'{idx:06d}_Doc1_00-00-00'
    timepoint_path = tmp_path / ds_name
    timepoint_path.mkdir()
    
    # copy - all ch
    for ch in all_channels:
        # make names
        ch_sfx = channels_sfx[ch]
        static_file_frame = tmpl_file_prfx + ch_sfx + static_file_frame_sfx 
        timela_file_frame = tmpl_file_prfx + ch_sfx + timela_file_frame_sfx 

        # copy static file:
        static_file_ch = static_file.replace(channels_sfx[ref_ch], ch_sfx)
        shutil.copy(static_file_ch, timepoint_path / static_file_frame)

        # copy timelapse file
        timelapse_file_ch = timelapse_file.replace(channels_sfx[ref_ch], ch_sfx)
        shutil.copy(timelapse_file_ch, timepoint_path / timela_file_frame)
    
    # make bat file for aligning 2 timeframes
    bat_file = make_bat(tmp_path, ds_name, idx, proc_bat_fn='proc_bgr.bat')
    
    
    # run bat file and wait for it to finish
    p = subprocess.Popen(str(bat_file), shell=True)
    p.wait()
    
    # read the file with the coordinates at the timepoint_path+'_A' / 'algn' / 'tile_000' / 'scanPos.txt'
    pos_file = tmp_path / f'{ds_name}_A' / 'algn' / 'tile_000' / 'scanPos.txt'
    with open(pos_file, 'rt') as f:
        lines = f.readlines()
    
    coords += lines[1]
    
    
# add the coordinates of the sttaic file (lines[0]) to the end of the coords
coords += lines[0]

# save the coords to a file in the tmp_path
coords_file = tmp_path / 'scanPos.txt'
with open(coords_file, 'wt') as f:
    f.write(coords)
    
    
# make joined dataset 000000_Doc1_00-00-01
ds_name_all = '000000_Doc1_00-00-01'
# timela_file_frame_all = r'00-00-01_Doc1_PMT - PMT [6RED] _C02_Time Time%04d.ome.tif'

all_file_prfx = '00-00-01_Doc1_PMT - PMT '
timela_all_file_frame_sfx = r' Time%04d.ome.tif'


# go through all the timepoints and move timelapse files to the new folder
all_timepoint_path = tmp_path / ds_name_all
all_timepoint_path.mkdir()

for idx, timelapse_file in enumerate(timelapse_files):
    ds_name = f'{idx:06d}_Doc1_00-00-00'
    timepoint_path = tmp_path / ds_name
    
    # copy - all ch
    for ch in all_channels:
        # make names
        ch_sfx = channels_sfx[ch]
        
        timela_file_frame = tmpl_file_prfx + ch_sfx + timela_file_frame_sfx
        timela_file_frame_all = all_file_prfx + ch_sfx + timela_all_file_frame_sfx % idx
        
        shutil.move(timepoint_path / timela_file_frame, all_timepoint_path / timela_file_frame_all)

last_idx = len(timelapse_files)
    
for ch in all_channels:
    # make names
    ch_sfx = channels_sfx[ch]

    # move static file to the new folder as well, the last one
    static_file_frame = tmpl_file_prfx + ch_sfx + static_file_frame_sfx 
    timela_file_frame_all_last = all_file_prfx + ch_sfx + timela_all_file_frame_sfx % last_idx
    shutil.move(timepoint_path / static_file_frame, all_timepoint_path / timela_file_frame_all_last)

# create reposotitioning bat file
repos_bat = make_repos_bat(tmp_path, ds_name_all, repos_bat_fn='repos_bgr.bat')

# run the repos.bat file
p = subprocess.Popen(str(repos_bat), shell=True)
p.wait()

# in the end one might want to move the sttatic timeframe to replicate the timelapse one at each timepoint as an addiitonal channel
out_path = all_timepoint_path.parent /(all_timepoint_path.stem+'_A')
all_tf = True

if all_tf:
    # copy - all ch
    for ch in all_channels:
        # make names
        ch_sfx = channels_sfx[ch]
        
        timela_file_frame_all = all_file_prfx + ch_sfx + timela_all_file_frame_sfx % last_idx
        
        static_tf_last = out_path / timela_file_frame_all
        
        for idx, timelapse_file in enumerate(timelapse_files):
            static_ch_sfx = ch_sfx.replace(' _C0', ' _C1')
            static_file_frame_all = all_file_prfx + static_ch_sfx + timela_all_file_frame_sfx % idx
            
            # copy static file (out_path / timela_file_frame_all) % (last_idx) to each timepoint
            shutil.copy(static_tf_last, out_path / static_file_frame_all)
        
        # remove the last file  -static_tf_last
        os.remove(static_tf_last)
        
# move the whole out_path folder to the final destination
timelapse_path_pl = pl.Path(timelapse_path)
final_path = timelapse_path_pl.parent / (timelapse_path_pl.stem + '_aligned')

if final_path.exists():
    print(f'{final_path} already exists, aborting'
          f'Please remove it first and rerun the script'
          f'You can find the aligned data in {out_path}')
else:
    shutil.move(out_path, final_path)
    print(f'Aligned data saved to {final_path}')
    
    #remove the tmp folder
    shutil.rmtree(tmp_path)

Aligned data saved to g:\2PM\Amandine\250408_Doc1_11-01-54_aligned


# v2

In [6]:
def parse_stack_fn(fn_or_path):
    fn = pl.Path(fn_or_path).name.replace('.ome.tif', '')
    
    parts = fn.split(' ')
    
    if len(parts) != 6:
        raise ValueError(f'Expected filename in format <hh>-<mm>-<ss>_Doc1_PMT - PMT [<ch_name>] _C<ch_idx>_Time Time<t_idx:004d>.ome.tif, received "{fn}"+.ome.tif')
    
    p_time, _, _, p_ch, p_ch_idx, p_t = parts
    
    time_str = p_time.replace('_Doc1_PMT', '')
    ch_name = p_ch[1:-1]
    
    ch_idx = int(p_ch_idx.replace('_C', '').replace('_Time', ''))
    t_idx = int(p_t.replace('Time', ''))
    
    d = {
        'time_str': time_str,
        'ch_name': ch_name,
        'ch_idx': ch_idx,
        't_idx': t_idx
    }
    
    return d

d = parse_stack_fn(r'd:/f/f/14-25-31_Doc1_PMT - PMT [5GREEN] _C00_Time Time0031.ome.tif')

In [33]:
# v2 [WIP]: Blue, Green & Red & FarRed, channels, alignebt by red, input given by parameters

def align_timelapse_to_static(timelapse_path:str, static_path:str, ref_ch:int=2):
    # Path to the folder containing the timelapse images
    #timelapse_path = r'g:\2PM\Amandine\250408_Doc1_11-01-54'
    # Path to the folder containing the static images
    #static_path = r'g:\2PM\Amandine\250408_Doc1_11-01-55'
    timelapse_path = str(timelapse_path)
    static_path = str(static_path)
    
    # 1. the channel ids - 00, 01, 02 - are the enabled only. true valeus have to be obtained from available files
    # 2. when searching with glob - only 2BLUE etc should be used
    channels_sfx_test = {
        0: '2BLUE', 
        1: '5GREEN', 
        2: '6RED',
        3: '7FarRED',
    }
    
#     channels_sfx = {
#         0: '[2BLUE] _C00_Time', 
#         1: '[5GREEN] _C01_Time', 
#         2: '[6RED] _C02_Time',
#         3: '[7FarRED] _C03_Time',
#     }
    
    channels_sfx = {}
    for ch_idx, ch_n in channels_sfx_test.items():
        test_files = glob.glob(static_path + f'/*{ch_n}*.tif')
        test_files.sort()
        if len(test_files) == 0: 
            continue
        test_fn = test_files[0]
        
        fn_info = parse_stack_fn(test_fn)
        ch_name = fn_info['ch_name']
        ch_idx = fn_info['ch_idx']
        
        channels_sfx[ch_idx] = f'[{ch_name}] _C{ch_idx:02d}_Time'
        
    
    align_channel_suffix = channels_sfx[ref_ch].replace('_Time', '').replace(' ', '*').replace('[', '').replace(']', '')  #'6RED*_C02'
    # list files in static folder

    static_files = glob.glob(static_path + f'/*{align_channel_suffix}*.tif')
    static_files.sort()
    #print(static_files)

    # list files in timelapse folder


    timelapse_files = glob.glob(timelapse_path + f'/*{align_channel_suffix}*.tif')
    timelapse_files.sort()
    #print(timelapse_files)


    assert len(static_files) == 1, 'There should be only one static image'

    static_file = static_files[0]

    # make tmp dir
    tmp_path = pl.Path(timelapse_files[0]).parent.parent / 'tmp'
    if tmp_path.exists():
        shutil.rmtree(tmp_path)

    tmp_path.mkdir()

    all_channels = list(channels_sfx.keys())
    # make name templates:
    # static_file_frame = r'00-00-00_Doc1_PMT - PMT [6RED] _C02_Time Time0000.ome.tif'
    # timela_file_frame = r'00-00-00_Doc1_PMT - PMT [6RED] _C02_Time Time0001.ome.tif'

    tmpl_file_prfx = '00-00-00_Doc1_PMT - PMT '
    static_file_frame_sfx = ' Time0000.ome.tif'
    timela_file_frame_sfx = ' Time0001.ome.tif'


    coords = ''
    for idx, timelapse_file in enumerate(timelapse_files):
        # make dir for each timepoint

        ds_name = f'{idx:06d}_Doc1_00-00-00'
        timepoint_path = tmp_path / ds_name
        timepoint_path.mkdir()

        # copy - all ch
        for ch in all_channels:
            # make names
            ch_sfx = channels_sfx[ch]
            static_file_frame = tmpl_file_prfx + ch_sfx + static_file_frame_sfx 
            timela_file_frame = tmpl_file_prfx + ch_sfx + timela_file_frame_sfx 

            # copy static file:
            static_file_ch = static_file.replace(channels_sfx[ref_ch], ch_sfx)
            shutil.copy(static_file_ch, timepoint_path / static_file_frame)

            # copy timelapse file
            timelapse_file_ch = timelapse_file.replace(channels_sfx[ref_ch], ch_sfx)
            shutil.copy(timelapse_file_ch, timepoint_path / timela_file_frame)

        # make bat file for aligning 2 timeframes
        bat_file = make_bat(tmp_path, ds_name, idx, proc_bat_fn='proc_grf.bat')


        # run bat file and wait for it to finish
        p = subprocess.Popen(str(bat_file), shell=True)
        p.wait()

        # read the file with the coordinates at the timepoint_path+'_A' / 'algn' / 'tile_000' / 'scanPos.txt'
        pos_file = tmp_path / f'{ds_name}_A' / 'algn' / 'tile_000' / 'scanPos.txt'
        with open(pos_file, 'rt') as f:
            lines = f.readlines()

        coords += lines[1]


    # add the coordinates of the sttaic file (lines[0]) to the end of the coords
    coords += lines[0]

    # save the coords to a file in the tmp_path
    coords_file = tmp_path / 'scanPos.txt'
    with open(coords_file, 'wt') as f:
        f.write(coords)


    # make joined dataset 000000_Doc1_00-00-01
    ds_name_all = '000000_Doc1_00-00-01'
    # timela_file_frame_all = r'00-00-01_Doc1_PMT - PMT [6RED] _C02_Time Time%04d.ome.tif'

    all_file_prfx = '00-00-01_Doc1_PMT - PMT '
    timela_all_file_frame_sfx = r' Time%04d.ome.tif'


    # go through all the timepoints and move timelapse files to the new folder
    all_timepoint_path = tmp_path / ds_name_all
    all_timepoint_path.mkdir()

    for idx, timelapse_file in enumerate(timelapse_files):
        ds_name = f'{idx:06d}_Doc1_00-00-00'
        timepoint_path = tmp_path / ds_name

        # copy - all ch
        for ch in all_channels:
            # make names
            ch_sfx = channels_sfx[ch]

            timela_file_frame = tmpl_file_prfx + ch_sfx + timela_file_frame_sfx
            timela_file_frame_all = all_file_prfx + ch_sfx + timela_all_file_frame_sfx % idx

            shutil.move(timepoint_path / timela_file_frame, all_timepoint_path / timela_file_frame_all)

    last_idx = len(timelapse_files)

    for ch in all_channels:
        # make names
        ch_sfx = channels_sfx[ch]

        # move static file to the new folder as well, the last one
        static_file_frame = tmpl_file_prfx + ch_sfx + static_file_frame_sfx 
        timela_file_frame_all_last = all_file_prfx + ch_sfx + timela_all_file_frame_sfx % last_idx
        shutil.move(timepoint_path / static_file_frame, all_timepoint_path / timela_file_frame_all_last)

    # create reposotitioning bat file
    repos_bat = make_repos_bat(tmp_path, ds_name_all, repos_bat_fn='repos_grf.bat')

    # run the repos.bat file
    p = subprocess.Popen(str(repos_bat), shell=True)
    p.wait()

    # in the end one might want to move the sttatic timeframe to replicate the timelapse one at each timepoint as an addiitonal channel
    out_path = all_timepoint_path.parent /(all_timepoint_path.stem+'_A')
    all_tf = True

    if all_tf:
        # copy - all ch
        for ch in all_channels:
            # make names
            ch_sfx = channels_sfx[ch]

            timela_file_frame_all = all_file_prfx + ch_sfx + timela_all_file_frame_sfx % last_idx

            static_tf_last = out_path / timela_file_frame_all

            for idx, timelapse_file in enumerate(timelapse_files):
                static_ch_sfx = ch_sfx.replace(' _C0', ' _C1')
                static_file_frame_all = all_file_prfx + static_ch_sfx + timela_all_file_frame_sfx % idx

                # copy static file (out_path / timela_file_frame_all) % (last_idx) to each timepoint
                shutil.copy(static_tf_last, out_path / static_file_frame_all)

            # remove the last file  -static_tf_last
            os.remove(static_tf_last)

    # move the whole out_path folder to the final destination
    timelapse_path_pl = pl.Path(timelapse_path)
    final_path = timelapse_path_pl.parent / (timelapse_path_pl.stem + '_aligned')

    if final_path.exists():
        print(f'{final_path} already exists, aborting'
              f'Please remove it first and rerun the script'
              f'You can find the aligned data in {out_path}')
    else:
        shutil.move(out_path, final_path)
        print(f'Aligned data saved to {final_path}')

        #remove the tmp folder
        shutil.rmtree(tmp_path)

In [35]:
def extract_timelapse_static_block(ds_path, ds_id, static_timelapse_root, block_sz, block_idx, static_idx=-1):
    if static_idx==-1:
        static_idx = block_sz - 1
        
    channels_sfx = {
        0: '2BLUE',
        1: '5GREEN',
        2: '6RED',
        3: '7FarRED',
    }
    
    channels_files = {}
    
    for ch_idx, ch_sfx in channels_sfx.items():
        ch_sfx_s = ch_sfx.replace(' ', '*')
        fns = glob.glob(ds_path + f'/*{ch_sfx_s}*.tif')
        fns.sort()
        if len(fns) == 0:
            continue
            
        #print(ch_sfx, ch_idx, fns)
        channels_files[ch_idx] = fns
        
    #print(channels_files)
    
    # get n time-frames
    #print(channels_files)
    ch_idx = next(iter(channels_files.keys()))
    #print(f'ch_idx = {ch_idx}')
    n_t = len(channels_files[ch_idx])
    
    f_info_d = parse_stack_fn(channels_files[ch_idx][0])
    # get time stampt = part before _Doc1_PMT
    time_str = f_info_d['time_str']
    
    # create dummy timestamp for proc
    # 30-block_idx-s(00)/tl(01)
    
    t_str_static = f'{ds_id:02d}-{block_idx:02d}-00'
    t_str_timlap = f'{ds_id:02d}-{block_idx:02d}-01'
    
    static_timelapse_root = pl.Path(static_timelapse_root)
    
    timelapse_path = static_timelapse_root / ('000000_Doc1_'+t_str_timlap)
    static_path = static_timelapse_root / ('000000_Doc1_'+t_str_static)

    # 1 . make dirs
    timelapse_path.mkdir(parents=True, exist_ok=True)
    static_path.mkdir(parents=True, exist_ok=True)

    # 2. loop relevant timespan

    for fns in channels_files.values():
        tgt_t_idx = -1
        for bt_idx in range(block_sz):
            t_idx = bt_idx + block_idx * block_sz
            src_file = fns[t_idx]

            is_static = bt_idx == static_idx

            # get fn info:
            f_info_d = parse_stack_fn(src_file)
            # get time stampt = part before _Doc1_PMT
            ch_name = f_info_d['ch_name']
            ch_idx = f_info_d['ch_idx']

            if is_static:
                stat_tgt_t_idx = 0
                tgt_file = static_path / (f'{t_str_static}_Doc1_PMT - PMT [{ch_name}] _C{ch_idx:02d}_Time Time{stat_tgt_t_idx:04d}.ome.tif')
            else:
                tgt_t_idx += 1
                tgt_file = timelapse_path / (f'{t_str_timlap}_Doc1_PMT - PMT [{ch_name}] _C{ch_idx:02d}_Time Time{tgt_t_idx:04d}.ome.tif')

            shutil.copy(src_file, tgt_file)

            # OPT: move and then move back after processing

    return timelapse_path, static_path

In [36]:
ds_paths = [
    r'g:\2PM\Amandine\all_25_04\ChP01-25\250417_Doc1_13-02-30\\',
    r'g:\2PM\Amandine\all_25_04\ChP01-25\250417_Doc1_13-33-48\\',

    r'g:\2PM\Amandine\all_25_04\ChP02-25\250417_Doc1_15-35-10\\',
    r'g:\2PM\Amandine\all_25_04\ChP02-25\250417_Doc1_16-09-55\\',

    r'g:\2PM\Amandine\all_25_04\ChP04-25\250408_Doc1_14-25-31\\',
    r'g:\2PM\Amandine\all_25_04\ChP04-25\250408_Doc1_13-50-11\\',

    ]
static_timelapse_root=r'g:\2PM\Amandine\\'
block_sz = 8

#block_idx = 0

for ds_id_p,  ds_path in enumerate(ds_paths):

    for block_idx in range(0, 4):
        timelapse_path, static_path = extract_timelapse_static_block(ds_path, ds_id_p+30, static_timelapse_root, block_sz, block_idx)
        print(timelapse_path, static_path)
        align_timelapse_to_static(timelapse_path, static_path, ref_ch=2)


g:\2PM\Amandine\000000_Doc1_30-00-01 g:\2PM\Amandine\000000_Doc1_30-00-00
g:\2PM\Amandine\000000_Doc1_30-00-01_aligned already exists, abortingPlease remove it first and rerun the scriptYou can find the aligned data in g:\2PM\Amandine\tmp\000000_Doc1_00-00-01_A
g:\2PM\Amandine\000000_Doc1_30-01-01 g:\2PM\Amandine\000000_Doc1_30-01-00
Aligned data saved to g:\2PM\Amandine\000000_Doc1_30-01-01_aligned
g:\2PM\Amandine\000000_Doc1_30-02-01 g:\2PM\Amandine\000000_Doc1_30-02-00
Aligned data saved to g:\2PM\Amandine\000000_Doc1_30-02-01_aligned
g:\2PM\Amandine\000000_Doc1_30-03-01 g:\2PM\Amandine\000000_Doc1_30-03-00
Aligned data saved to g:\2PM\Amandine\000000_Doc1_30-03-01_aligned
g:\2PM\Amandine\000000_Doc1_31-00-01 g:\2PM\Amandine\000000_Doc1_31-00-00
Aligned data saved to g:\2PM\Amandine\000000_Doc1_31-00-01_aligned
g:\2PM\Amandine\000000_Doc1_31-01-01 g:\2PM\Amandine\000000_Doc1_31-01-00
Aligned data saved to g:\2PM\Amandine\000000_Doc1_31-01-01_aligned
g:\2PM\Amandine\000000_Doc1_31-02

In [73]:
d

{'time_str': '14-25-31', 'ch_name': '5GREEN', 'ch_idx': 0, 't_idx': 31}

In [38]:
extract_timelapse_static_block(r'g:\2PM\Amandine\all_25_04\ChP04-25\250408_Doc1_14-25-31\\', '', '', 8, 0)

2BLUE []
5GREEN ['g:\\2PM\\Amandine\\all_25_04\\ChP04-25\\250408_Doc1_14-25-31\\14-25-31_Doc1_PMT - PMT [5GREEN] _C00_Time Time0000.ome.tif', 'g:\\2PM\\Amandine\\all_25_04\\ChP04-25\\250408_Doc1_14-25-31\\14-25-31_Doc1_PMT - PMT [5GREEN] _C00_Time Time0001.ome.tif', 'g:\\2PM\\Amandine\\all_25_04\\ChP04-25\\250408_Doc1_14-25-31\\14-25-31_Doc1_PMT - PMT [5GREEN] _C00_Time Time0002.ome.tif', 'g:\\2PM\\Amandine\\all_25_04\\ChP04-25\\250408_Doc1_14-25-31\\14-25-31_Doc1_PMT - PMT [5GREEN] _C00_Time Time0003.ome.tif', 'g:\\2PM\\Amandine\\all_25_04\\ChP04-25\\250408_Doc1_14-25-31\\14-25-31_Doc1_PMT - PMT [5GREEN] _C00_Time Time0004.ome.tif', 'g:\\2PM\\Amandine\\all_25_04\\ChP04-25\\250408_Doc1_14-25-31\\14-25-31_Doc1_PMT - PMT [5GREEN] _C00_Time Time0005.ome.tif', 'g:\\2PM\\Amandine\\all_25_04\\ChP04-25\\250408_Doc1_14-25-31\\14-25-31_Doc1_PMT - PMT [5GREEN] _C00_Time Time0006.ome.tif', 'g:\\2PM\\Amandine\\all_25_04\\ChP04-25\\250408_Doc1_14-25-31\\14-25-31_Doc1_PMT - PMT [5GREEN] _C00_Time Ti

In [65]:
align_timelapse_to_static(timelapse_path, r'g:\2PM\Amandine\all_25_04\ChP04-25\250408_Doc1_14-25-31')

{0: '[5GREEN] _C00_Time', 1: '[6RED] _C01_Time', 2: '[7FarRED] _C02_Time'}
